In [1]:
!pip uninstall -y numpy pyarrow pandas datasets transformers faiss faiss-cpu

Found existing installation: numpy 2.3.5
Uninstalling numpy-2.3.5:
  Successfully uninstalled numpy-2.3.5
Found existing installation: pyarrow 21.0.0
Uninstalling pyarrow-21.0.0:
  Successfully uninstalled pyarrow-21.0.0
Found existing installation: pandas 2.3.2
Uninstalling pandas-2.3.2:
  Successfully uninstalled pandas-2.3.2
Found existing installation: transformers 4.56.2
Uninstalling transformers-4.56.2:
  Successfully uninstalled transformers-4.56.2


In [2]:
!pip install -q \
torch==2.2.2 \
torchvision==0.17.2 \
torchaudio==2.2.2 \
--index-url https://download.pytorch.org/whl/cu121

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xfields 0.25.15 requires pandas, which is not installed.
xtrack 0.101.4 requires pandas>=2.0, which is not installed.
xcoll 0.9.11 requires pandas>=1.4, which is not installed.
fair 2.2.2 requires pandas, which is not installed.
pynetcf 0.5.1 requires pandas, which is not installed.
ismn 1.5.2 requires pandas, which is not installed.
pytesmo 0.18.0 requires pandas>=0.23.0, which is not installed.
ipytablewidgets 0.3.2 requires pandas<3.0.0,>=1.0.0, which is not installed.
tensorflow-datasets 4.9.9 requires pyarrow, which is not installed.
holoviews 1.21.0 requires pandas>=1.3, which is not installed.
resipy 3.5.4 requires pandas, which is not installed.
metpy 1.7.1 requires pandas>=1.4.0, which is not installed.
pygeogrids 0.5.2 requires pandas, which is not installed.
hvplot 0.12.1 requires pandas>=1.3, which is 

In [3]:
!pip install -q \
transformers==4.30.0 \
datasets==2.14.6 \
faiss-cpu==1.7.4 \
numpy==1.26.4 \
pandas==2.1.4 \
pyarrow==12.0.1 \
sentencepiece \
accelerate \
evaluate \
sacrebleu \
scikit-learn \
tqdm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grader-service 0.12.0 requires pandas>=2.2.3, but you have pandas 2.1.4 which is incompatible.
jax 0.9.0.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.9.0.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytesmo 0.18.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
obspy 1.4.2 requires sqlalchemy<2, but you have sqlalchemy 2.0.43 which is incompatible.
pygimli 1.5.4 requires numpy>=2.1, but you have numpy 1.26.4 which is incompatible.
tetgen 0.6.7 requires numpy<3,>=2, but you have numpy 1.26.4 which is incompatible.
xarray 2025.9.0 requires pandas>=2.2, but you have pandas 2.1.4 which is incompatible.
torchtext 0.18.0 requires torch>=2.3.0, but you have torch 2.2.2+cu121 which is incompatible.
opencv-python-headless 4.12.0.88 require

In [4]:
import torch
import sacrebleu
import nltk

from datasets import load_dataset

from transformers import (

    RagTokenizer,
    RagRetriever,
    RagSequenceForGeneration,
    TrainingArguments,
    Trainer,
    default_data_collator
)

from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

/opt/micromamba/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/micromamba/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/opt/micromamba/lib/python3.11/site-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please update the gencode to avoid compatibility violation

cuda


In [5]:
dataset = load_dataset(
    "search_qa",
    "train_test_val"
)

train_ds = dataset["train"].select(range(80000))

val_ds = dataset["validation"]

test_ds = dataset["test"]

print(len(train_ds))
print(len(val_ds))
print(len(test_ds))

80000
21613
43228


In [6]:
def preprocess(example):
    return {
        "source":str(example["answer"]),
        "target":str(example["question"])
    }

train_ds = train_ds.map(preprocess)
val_ds = val_ds.map(preprocess)
test_ds = test_ds.map(preprocess)

In [7]:
tokenizer_seq = RagTokenizer.from_pretrained(
    "facebook/rag-sequence-nq"
)

retriever_seq = RagRetriever.from_pretrained(
    "facebook/rag-sequence-nq",
    index_name="compressed",
    use_dummy_dataset=False
)

rag_sequence = RagSequenceForGeneration.from_pretrained(
    "facebook/rag-sequence-nq",
    retriever=retriever_seq
).to(device)

rag_sequence.config.n_docs = 10

print("RAG SEQUENCE LOADED")

/opt/micromamba/lib/python3.11/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizer'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'RagTokenizer'. 
The class this function is called from is 'DPRQuestionEncoderTokenizerFast'.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called 

RAG SEQUENCE LOADED


In [8]:
def tokenize_function(examples):

    model_inputs = tokenizer_seq.question_encoder(
        examples["source"],
        truncation=True,
        padding="max_length",
        max_length=32
    )

    labels = tokenizer_seq.generator(
        examples["target"],
        truncation=True,
        padding="max_length",
        max_length=32
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [9]:
train_seq = train_ds.map(
    tokenize_function,
    batched=True
)

val_seq = val_ds.map(
    tokenize_function,
    batched=True
)

test_seq = test_ds.map(
    tokenize_function,
    batched=True
)

In [10]:
class RagTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False
    ):

        outputs = model(**inputs)

        loss = outputs.loss.mean()

        return (
            (loss,outputs)
            if return_outputs
            else loss
        )

In [11]:
for param in rag_sequence.question_encoder.parameters():
    param.requires_grad = False

training_args = TrainingArguments(
    output_dir="./rag_sequence",
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    warmup_steps=500,
    weight_decay=0.01,
    fp16=True,
    evaluation_strategy="no",
    save_steps=1000,
    logging_steps=100,
    save_total_limit=2,
    report_to="none"
)

In [12]:
trainer = RagTrainer(
    model=rag_sequence,
    args=training_args,
    train_dataset=train_seq,
    tokenizer=tokenizer_seq.generator,
    data_collator=default_data_collator
)

In [13]:
trainer.train()

/opt/micromamba/lib/python3.11/site-packages/transformers/optimization.py:411: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


Step,Training Loss
100,151.327900
200,90.978800
300,79.823900
400,76.360700
500,74.036500
600,72.632500
700,72.436100
800,69.526500
900,68.658000
1000,68.925600


TrainOutput(global_step=10000, training_loss=63.54427631835937, metrics={'train_runtime': 50423.6851, 'train_samples_per_second': 3.173, 'train_steps_per_second': 0.198, 'total_flos': 1.344844333056e+16, 'train_loss': 63.54427631835937, 'epoch': 2.0})

In [14]:
rag_sequence.save_pretrained(
    "./rag_sequence_final"
)

tokenizer_seq.save_pretrained(
    "./rag_sequence_final"
)

print("RAG SEQUENCE SAVED")

RAG SEQUENCE SAVED


In [15]:
# tokenizer_seq = RagTokenizer.from_pretrained(
#     "./rag_sequence_final"
# )

# retriever_seq = RagRetriever.from_pretrained(
#     "facebook/rag-sequence-nq",
#     index_name="compressed",
#     use_dummy_dataset=False
# )

# rag_sequence = RagSequenceForGeneration.from_pretrained(
#     "./rag_sequence_final",
#     retriever=retriever_seq
# ).to(device)

# rag_sequence.eval()

In [16]:
examples = [
    "Albert Einstein",
    "The Beatles",
    "Leonardo da Vinci",
    "Michael Jordan",
    "Marie Curie",
    "William Shakespeare"
]

rag_sequence.eval()

for entity in examples:

    inputs = tokenizer_seq.question_encoder(
        entity,
        return_tensors="pt"
    )

    inputs = {
        k:v.to(device)
        for k,v in inputs.items()
        if k!="token_type_ids"
    }

    with torch.no_grad():
        outputs = rag_sequence.generate(
            **inputs,
            num_beams=4,
            max_new_tokens=50
        )

    prediction = tokenizer_seq.generator.batch_decode(
        outputs,
        skip_special_tokens=True
    )[0]
    print("\n"+"="*100)
    print("ENTITY:",entity)
    print("\nRAG SEQUENCE:")
    print(prediction)


ENTITY: Albert Einstein

RAG SEQUENCE:
In 1922 he was awarded the Nobel Prize in Physics

ENTITY: The Beatles

RAG SEQUENCE:
"Sgt. Pepper's Lonely Hearts Club Band"

ENTITY: Leonardo da Vinci

RAG SEQUENCE:
This Florentine's "Mona Lisa" is seen here

ENTITY: Michael Jordan

RAG SEQUENCE:
He's the only player in NBA history to win 2 Olympic gold medals

ENTITY: Marie Curie

RAG SEQUENCE:
In 1932 this Polish chemist founded the Radium Institute in Krakow

ENTITY: William Shakespeare

RAG SEQUENCE:
"Hamlet"


In [17]:
from sacrebleu.metrics import BLEU

def evaluate_bleu1(
    model,
    tokenizer,
    dataset,
    n_samples=1000
):

    model.eval()

    preds = []
    refs = []

    subset = dataset.select(
        range(min(n_samples, len(dataset)))
    )

    for ex in subset:

        source = ex["source"]
        target = ex["target"]

        inputs = tokenizer.question_encoder(
            source,
            return_tensors="pt",
            truncation=True,
            max_length=32
        )

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
            if k != "token_type_ids"
        }

        with torch.no_grad():

            outputs = model.generate(
                **inputs,
                num_beams=4,
                max_new_tokens=50
            )

        pred = tokenizer.generator.batch_decode(
            outputs,
            skip_special_tokens=True
        )[0]

        preds.append(pred)
        refs.append(target)

    bleu1_metric = BLEU(
        max_ngram_order=1
    )

    bleu1 = bleu1_metric.corpus_score(
        preds,
        [refs]
    )

    return bleu1.score

In [23]:
references = []
predictions = []

batch = val_ds[:10000]

for source, target in zip(batch["source"], batch["target"]):
    
    inputs = tokenizer_seq.question_encoder(
        source,
        return_tensors="pt",
        truncation=True,
        max_length=32
    )

    inputs = {
        k:v.to(devicsourcee)
        for k,v in inputs.items()
        if k!="token_type_ids"
    }

    with torch.no_grad():

        outputs = rag_sequence.generate(
            **inputs,
            num_beams=4,
            max_new_tokens=50
        )

    pred = tokenizer_seq.generator.decode(
        outputs[0],
        skip_special_tokens=True
    )

    references.append(target)
    predictions.append(pred)

print("Exporting")

with open("references.txt","w") as f:
    for q in references:
        f.write(q.strip()+"\n")

with open("hypotheses.txt","w") as f:
    for q in predictions:
        f.write(q.strip()+"\n")

Exporting


In [19]:

val_b1 = evaluate_bleu1(
    rag_sequence,
    tokenizer_seq,
    val_ds
)

print("VAL B-1:",val_b1)

VAL B-1: 8.549218614575075


In [24]:
import subprocess
import re

cmd = [
    "python",
    "Answerability-Metric/answerability_score.py",
    "--data_type", "squad",
    "--ref_file", "references.txt",
    "--hyp_file", "hypotheses.txt",
    "--ner_weight", "0.41",
    "--qt_weight", "0.20",
    "--re_weight", "0.36",
    "--delta", "0.66",
    "--ngram_metric", "Bleu_1"
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print(result.stdout)

# Extract the official scores

text = result.stderr

qbleu1 = float(
    re.search(
        r"Mean Answerability Score Across Questions:\s*([\d.]+)",
        result.stderr
    ).group(1)
)

bleu1 = float(
    re.search(
        r"N-gram Score:\s*([\d.]+)",
        result.stderr
    ).group(1)
)

print(f"Official Q-BLEU-1: {100*qbleu1:.2f}")
print(f"Official BLEU-1: {100*bleu1:.2f}")


Official Q-BLEU-1: 18.10
Official BLEU-1: 10.00


a

In [30]:
rag_sequence.config.num_return_sequences

1

In [31]:
rag_sequence.config.n_docs

10

In [44]:
print(test_ds[:10]["source"])

['England', 'Miley Cyrus', 'the skin', '48', 'dribbling', 'heart', 'the equator', 'Care Bears', 'H2O', '98']
